# 03 Taxonomic classification (ASVs)

Input: representative sequences and feature table from `02_denoise.ipynb`

Output (`processed_ignored/03_taxonomy/`): `taxonomy.qza`, taxonomy-filtered table and sequences (`table_fungi_filtered.qza`, `rep_set_fungi_filtered.qza`) used for all downstream analyses

---
extended version

# 03 Taxonomic Classification (ASVs)

**Creator:** Jingxi

**Input:**

`processed_ignored/02_denoise/`
- representative ASV sequences (`dada2_rep_set.qza`)
- ASV feature table (`dada2_table.qza`)\

`raw/`
- UNITE fungal classifier (`unite_classifier_dynamic_fungi.qza`)

`raw/uniteDB/`
- UNITE reference sequences (`sequences.qza`)
- UNITE reference taxonomy (`taxonomy.qza`)

**Output:**

`processed_ignored/03_taxonomy/`
- ASV taxonomic assignments (`taxonomy.qza`)
- filtered fungal ASV feature table (`table_fungi_filtered.qza`)
- filtered fungal representative ASV sequences (`rep_set_fungi_filtered.qza`)\

`processed/03_taxonomy/`
- filtered fungal taxonomic bar plot (`taxa_barplot_fungi_filtered.qzv`)

**Intermediate files:**

`raw/uniteDB/`
- filtered UNITE reference sequences (`sequences-filtered.qza`)
- UNITE reference taxonomy with Species Hypothesis (SH) IDs removed (`taxonomy-no-SH.qza`)

The filtered table (`table_fungi_filtered.qza`) and representative sequences (`rep_set_fungi_filtered.qza`) are used for downstream ASV analyses.

## 0 set up

In [1]:
import pandas as pd
from qiime2 import Visualization, Artifact

%matplotlib inline

raw_dir = '../data/raw'
in_dir  = '../data/processed_ignored/02_denoise'
out_dir = '../data/processed_ignored/03_taxonomy'
vis_dir = '../data/processed/03_taxonomy'
unite_dir  = f'{raw_dir}/uniteDB'
classifier = f'{raw_dir}/unite_classifier_dynamic_fungi.qza'

In [2]:
!mkdir -p $out_dir $vis_dir

## 01 Classifier

Training our own naive Bayes classifier on UNITE (with RESCRIPt, section 2b below) and classifying with the eukaryote classifier both exceeded the 12 GB memory limit of JupyterHub. We therefore use the **pre-trained UNITE v10.0 (2025-02-19) dynamic fungi classifier** for QIIME 2 2026.7 from https://github.com/colinbrislawn/unite-train/releases.

In [5]:
#!qiime rescript get-unite-data \
    #--p-version 2025-02-19 \
   # --p-taxon-group eukaryotes \
    #--p-cluster-id dynamic \
    #--p-no-singletons \
   # --verbose \
   # --output-dir uniteDB

Saved FeatureData[Taxonomy] to: uniteDB/taxonomy.qza
Saved FeatureData[Sequence] to: uniteDB/sequences.qza


In [7]:
#!qiime taxa filter-seqs \
  #  --p-exclude Fungi_sp,mycota_sp,mycetes_sp \
   # --i-taxonomy uniteDB/taxonomy.qza \
   # --i-sequences uniteDB/sequences.qza \
    #--o-filtered-sequences uniteDB/sequences-filtered.qza

Saved FeatureData[Sequence] to: uniteDB/sequences-filtered.qza


In [8]:
#!qiime rescript edit-taxonomy \
    #--i-taxonomy uniteDB/taxonomy.qza \
   # --o-edited-taxonomy uniteDB/taxonomy-no-SH.qza \
    #--p-search-strings ';sh__.*' \
   # --p-replacement-strings '' \
    #--p-use-regex

Saved FeatureData[Taxonomy] to: uniteDB/taxonomy-no-SH.qza


In [ ]:
#!qiime feature-classifier fit-classifier-naive-bayes \
    #--i-reference-reads uniteDB/sequences-filtered.qza \
    #--i-reference-taxonomy uniteDB/taxonomy-no-SH.qza \
    #--o-classifier uniteDB/classifier.qza

download classifier

In [4]:
!wget -nc -O $classifier https://github.com/colinbrislawn/unite-train/releases/download/v10.0-2025-02-19-qiime2-2026.7/unite_ver2025-02-19_dynamic_fungi-Q2-2026.7.qza
!wget -nc -O $vis_dir/eval_unite_ver2025-02-19.qzv https://github.com/colinbrislawn/unite-train/releases/download/v10.0-2025-02-19-qiime2-2026.7/eval_unite_ver2025-02-19.qzv

--2026-11-09 02:24:28--  https://github.com/colinbrislawn/unite-train/releases/download/v10.0-2025-02-19-qiime2-2026.7/unite_ver2025-02-19_dynamic_fungi-Q2-2026.7.qza
Resolving github.com (github.com)... 20.250.119.67
Connecting to github.com (github.com)|20.250.119.67|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/448594227/441c9784-6a39-4f22-b832-794cb6ab72cf?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-09T02%3A03%3A39Z&rscd=attachment%3B+filename%3Dunite_ver2025-02-19_dynamic_fungi-Q2-2026.7.qza&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-11-09T01%3A03%3A23Z&ske=2026-11-09T02%3A03%3A39Z&sks=b&skv=2018-11-09&sig=B21AptWoSoC91BH%2B%2FlmXZpRhPSNP1%2B%2BTLcIG9AoI%2FUM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwi

In [ ]:
evaluate

In [ ]:
#!qiime rescript evaluate-fit-classifier \
    #--i-sequences uniteDB/sequences-filtered.qza   \
    #--i-taxonomy uniteDB/taxonomy-no-SH.qza \
    #--p-n-jobs 2 \
    #--o-classifier uniteDB/classifier.qza \
    #--o-evaluation uniteDB/classifier-evaluation.qzv \
    #--o-observed-taxonomy uniteDB/predicted-taxonomy.qza

#!qiime rescript evaluate-taxonomy \
  #--i-taxonomies uniteDB/taxonomy-no-SH.qza uniteDB/predicted-taxonomy.qza \
  #--p-labels ref-taxonomy predicted-taxonomy \
  #--o-taxonomy-stats uniteDB/both-taxonomy-evaluation.qzv

## 03 Taxonomy assignment

In [6]:
# n-jobs 1 and small batches to stay below the memory limit
!qiime feature-classifier classify-sklearn \
  --i-classifier $classifier \
  --i-reads $in_dir/dada2_rep_set.qza \
  --p-n-jobs 1 \
  --p-reads-per-batch 500 \
  --o-classification $out_dir/taxonomy.qza

Saved FeatureData[Taxonomy] to: ../data/processed_ignored/03_taxonomy/taxonomy.qza


In [7]:
!qiime metadata tabulate \
  --m-input-file $out_dir/taxonomy.qza \
  --o-visualization $vis_dir/taxonomy.qzv

Saved Visualization to: ../data/processed/03_taxonomy/taxonomy.qzv


In [8]:
Visualization.load(f'{vis_dir}/taxonomy.qzv')

<visualization: Visualization uuid: 5b048262-32b2-4571-859d-667c2bf0300d>

## 04 Filter by taxonomy

Keep ASVs classified at least to class level (`c__`); remove `Unassigned` and `unidentified`. This removes non-fungal and poorly classified sequences.

In [9]:
!qiime taxa filter-table \
  --i-table $in_dir/dada2_table.qza \
  --i-taxonomy $out_dir/taxonomy.qza \
  --p-include c__ \
  --p-exclude Unassigned,unidentified \
  --o-filtered-table $out_dir/table_fungi_filtered.qza

!qiime taxa filter-seqs \
  --i-sequences $in_dir/dada2_rep_set.qza \
  --i-taxonomy $out_dir/taxonomy.qza \
  --p-include c__ \
  --p-exclude Unassigned,unidentified \
  --o-filtered-sequences $out_dir/rep_set_fungi_filtered.qza

Saved FeatureTable[Frequency] to: ../data/processed_ignored/03_taxonomy/table_fungi_filtered.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/03_taxonomy/rep_set_fungi_filtered.qza


In [10]:
!qiime feature-table summarize \
  --i-table $out_dir/table_fungi_filtered.qza \
  --m-metadata-file $raw_dir/fungut_metadata.tsv \
  --o-feature-frequencies $out_dir/table_fungi_filtered_feature_frequencies.qza \
  --o-sample-frequencies $out_dir/table_fungi_filtered_sample_frequencies.qza \
  --o-summary $vis_dir/table_fungi_filtered.qzv

Saved ImmutableMetadata to: ../data/processed_ignored/03_taxonomy/table_fungi_filtered_feature_frequencies.qza
Saved ImmutableMetadata to: ../data/processed_ignored/03_taxonomy/table_fungi_filtered_sample_frequencies.qza
Saved Visualization to: ../data/processed/03_taxonomy/table_fungi_filtered.qzv


## 05 after filtering

In [11]:
t_before = Artifact.load(f'{in_dir}/dada2_table.qza').view(pd.DataFrame)
t_after  = Artifact.load(f'{out_dir}/table_fungi_filtered.qza').view(pd.DataFrame)
print('ASVs before:', t_before.shape[1], ' after:', t_after.shape[1])

comp = pd.DataFrame({'before': t_before.sum(axis=1), 'after': t_after.sum(axis=1)}).fillna(0)
comp['retained_%'] = comp['after'] / comp['before'] * 100
print(comp['retained_%'].describe().round(1))
comp.sort_values('retained_%').head(10)

ASVs before: 681  after: 575
count    150.0
mean      99.9
std        0.7
min       91.6
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: retained_%, dtype: float64


,before,after,retained_%
ERR5327557,17910.0,16402.0,91.580123
ERR5327418,33631.0,33449.0,99.458833
ERR5327591,37752.0,37552.0,99.470227
ERR5327503,50824.0,50584.0,99.527782
ERR5327566,20221.0,20152.0,99.658771
ERR5327475,35728.0,35615.0,99.683721
ERR5327596,21032.0,20968.0,99.695702
ERR5327550,35636.0,35562.0,99.792345
ERR5327465,36507.0,36446.0,99.832909
ERR5327198,32764.0,32720.0,99.865706


## 06 Taxanomy barplot

In [12]:
!qiime taxa barplot \
  --i-table $out_dir/table_fungi_filtered.qza \
  --i-taxonomy $out_dir/taxonomy.qza \
  --m-metadata-file $raw_dir/fungut_metadata.tsv \
  --o-visualization $vis_dir/taxa_barplot_fungi_filtered.qzv

Saved Visualization to: ../data/processed/03_taxonomy/taxa_barplot_fungi_filtered.qzv


In [13]:
Visualization.load(f'{vis_dir}/taxa_barplot_fungi_filtered.qzv')

<visualization: Visualization uuid: 30764f65-36cf-4ba6-a3b2-d8b6c014adb7>

## 07 Check for non-fungal sequences with a eukaryote reference

In [5]:
#Download and curate the UNITE eukaryote database
!qiime rescript get-unite-data \
    --p-version 2025-02-19 \
    --p-taxon-group eukaryotes \
    --p-cluster-id dynamic \
    --p-no-singletons \
    --output-dir $unite_dir

!qiime taxa filter-seqs \
    --p-exclude Fungi_sp,mycota_sp,mycetes_sp \
    --i-taxonomy $unite_dir/taxonomy.qza \
    --i-sequences $unite_dir/sequences.qza \
    --o-filtered-sequences $unite_dir/sequences-filtered.qza

!qiime rescript edit-taxonomy \
    --i-taxonomy $unite_dir/taxonomy.qza \
    --o-edited-taxonomy $unite_dir/taxonomy-no-SH.qza \
    --p-search-strings ';sh__.*' \
    --p-replacement-strings '' \
    --p-use-regex

Saved FeatureData[Taxonomy] to: ../data/raw/uniteDB/taxonomy.qza
Saved FeatureData[Sequence] to: ../data/raw/uniteDB/sequences.qza
Saved FeatureData[Sequence] to: ../data/raw/uniteDB/sequences-filtered.qza
Saved FeatureData[Taxonomy] to: ../data/raw/uniteDB/taxonomy-no-SH.qza


In [14]:
#classification with vsearch
!qiime feature-classifier classify-consensus-vsearch \
  --i-query $in_dir/dada2_rep_set.qza \
  --i-reference-reads $unite_dir/sequences-filtered.qza \
  --i-reference-taxonomy $unite_dir/taxonomy-no-SH.qza \
  --p-threads 1 \
  --o-classification $out_dir/taxonomy_vsearch_euk.qza \
  --o-search-results $out_dir/vsearch_euk_hits.qza

Saved FeatureData[Taxonomy] to: ../data/processed_ignored/03_taxonomy/taxonomy_vsearch_euk.qza
Saved FeatureData[BLAST6] to: ../data/processed_ignored/03_taxonomy/vsearch_euk_hits.qza


In [15]:
#Number of ASVs and reads per kingdom
tax_euk = Artifact.load(f'{out_dir}/taxonomy_vsearch_euk.qza').view(pd.DataFrame)
table   = Artifact.load(f'{in_dir}/dada2_table.qza').view(pd.DataFrame)

tax_euk['kingdom'] = tax_euk['Taxon'].str.split(';').str[0]
reads_per_asv = table.sum(axis=0)
summary = (tax_euk.assign(reads=reads_per_asv)
                  .groupby('kingdom')
                  .agg(ASVs=('Taxon', 'size'), reads=('reads', 'sum')))
summary['reads_%'] = (summary['reads'] / summary['reads'].sum() * 100).round(2)
summary.sort_values('reads', ascending=False)

,ASVs,reads,reads_%
kingdom,,,
k__Fungi,670,9037706.0,100.0
Unassigned,9,55.0,0.0
k__Metazoa,2,51.0,0.0
